In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/jps_coastal_erosion_full.csv")
df_clean = df.copy()

print("Dataset shape:", df_clean.shape)
df_clean.head()

Dataset shape: (144, 14)


,No,Coastal_Cell,District,Area_Location,Longitude_DMS,Latitude_DMS,Longitude_DD,Latitude_DD,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score,Total_Score,ISMP_2022_Category,NCES_2015_Category
0,1,C1,Besut,North of Kampung Teluk Budu,"102°32'47.8""E","5°50'18.7""N",102.546611,5.838528,4.94,5.00,3.4,17.00,1,1
1,2,C1,Besut,Breakwater near Sungai Besut river mouth,"102°33'26.7""E","5°50'03.3""N",102.557417,5.834250,0.00,0.00,3.2,0.00,3,2
2,3,C1,Besut,Pantai Bukit Keluang,"102°36'17.9""E","5°48'08.2""N",102.604972,5.802278,1.92,2.00,2.3,4.60,3,2
3,4,C1,Besut,Pantai Bukit Keluang,"102°36'35.9""E","5°47'56.6""N",102.609972,5.799056,0.94,0.94,1.4,1.32,3,3
4,5,C1,Besut,Pantai Bukit Keluang,"102°36'40.1""E","5°47'51.6""N",102.611139,5.797667,5.28,5.00,1.0,5.00,3,3


In [3]:
df_clean["Is_Protected"] = (
    df_clean["Annual_Erosion_Rate_m_yr"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("protected")
).astype(int)

df_clean["Is_Protected"].value_counts()

Is_Protected
0    116
1      9
Name: count, dtype: int64

In [3]:
print("Total records:", len(df_clean))
print("First JPS No:", df_clean["No"].min())
print("Last JPS No:", df_clean["No"].max())

print("Missing station numbers:",
      sorted(set(range(1, 145)) - set(df_clean["No"])))

print("Duplicate station numbers:",
      df_clean["No"].duplicated().sum())

Total records: 144
First JPS No: 1
Last JPS No: 144
Missing station numbers: []
Duplicate station numbers: 0


In [4]:
df_clean["Annual_Erosion_Rate_m_yr"] = pd.to_numeric(
    df_clean["Annual_Erosion_Rate_m_yr"],
    errors="coerce"
)

df_clean["Physical_Erosion_Score"] = pd.to_numeric(
    df_clean["Physical_Erosion_Score"],
    errors="coerce"
)

df_clean["Economic_Score"] = pd.to_numeric(
    df_clean["Economic_Score"],
    errors="coerce"
)

df_clean["Total_Score"] = pd.to_numeric(
    df_clean["Total_Score"],
    errors="coerce"
)

In [5]:
df_clean.dtypes

No                            int64
Coastal_Cell                    str
District                        str
Area_Location                   str
Longitude_DMS                   str
Latitude_DMS                    str
Longitude_DD                float64
Latitude_DD                 float64
Annual_Erosion_Rate_m_yr    float64
Physical_Erosion_Score      float64
Economic_Score              float64
Total_Score                 float64
ISMP_2022_Category              str
NCES_2015_Category            int64
Is_Protected                  int64
dtype: object

In [6]:
df_clean[df_clean["Is_Protected"] == 1][
    [
        "No",
        "Coastal_Cell",
        "Area_Location",
        "Annual_Erosion_Rate_m_yr",
        "Physical_Erosion_Score",
        "Economic_Score",
        "Total_Score",
        "Is_Protected"
    ]
]

,No,Coastal_Cell,Area_Location,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score,Total_Score,Is_Protected
16,36,C4,North of Kampung Pak Tuyu,NaN,NaN,3.3,0.0,1
17,37,C4,Kampung Pak Tuyu,NaN,NaN,2.7,0.0,1
18,38,C5,Kampung Seberang Takir,NaN,NaN,4.0,0.0,1
90,110,C9,Kampung Sura,NaN,NaN,2.1,0.0,1
91,111,C9,Kampung Sura,NaN,NaN,2.5,0.0,1
92,112,C9,Kampung Balai Besar,NaN,NaN,2.5,0.0,1
100,120,C10,Sand spit near Sungai Paka river mouth,NaN,NaN,1.1,0.0,1
118,138,C11,Pantai Kemasik (north),NaN,NaN,2.8,0.0,1
119,139,C11,Kampung Tengah,NaN,NaN,2.8,0.0,1


In [7]:
df_clean["Calculated_Total_Score"] = (
    df_clean["Physical_Erosion_Score"]
    * df_clean["Economic_Score"]
)

df_clean["Score_Difference"] = (
    df_clean["Calculated_Total_Score"]
    - df_clean["Total_Score"]
).abs()

df_clean["Score_Consistency"] = np.where(
    df_clean["Score_Difference"].fillna(0) <= 0.01,
    "Consistent",
    "Check"
)

df_clean[
    df_clean["Score_Consistency"] == "Check"
][
    [
        "No",
        "Coastal_Cell",
        "Area_Location",
        "Physical_Erosion_Score",
        "Economic_Score",
        "Total_Score",
        "Calculated_Total_Score"
    ]
]

,No,Coastal_Cell,Area_Location,Physical_Erosion_Score,Economic_Score,Total_Score,Calculated_Total_Score
108,128,C10,Petronas Kertih Refinery,5.0,3.1,9.3,15.5


In [8]:
print("Duplicate rows:", df_clean.duplicated().sum())
print("Duplicate coordinates:",
      df_clean.duplicated(
          subset=["Latitude_DD", "Longitude_DD"]
      ).sum())

Duplicate rows: 0
Duplicate coordinates: 0


In [9]:
print(
    "Latitude:",
    df_clean["Latitude_DD"].min(),
    "to",
    df_clean["Latitude_DD"].max()
)

print(
    "Longitude:",
    df_clean["Longitude_DD"].min(),
    "to",
    df_clean["Longitude_DD"].max()
)

Latitude: 4.21125 to 5.838528
Longitude: 102.546611 to 103.484222


In [10]:
invalid_coordinates = df_clean[
    ~df_clean["Latitude_DD"].between(-90, 90)
    |
    ~df_clean["Longitude_DD"].between(-180, 180)
]

invalid_coordinates

,No,Coastal_Cell,District,Area_Location,Longitude_DMS,Latitude_DMS,Longitude_DD,Latitude_DD,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score,Total_Score,ISMP_2022_Category,NCES_2015_Category,Is_Protected,Calculated_Total_Score,Score_Difference,Score_Consistency


In [11]:
df_clean.isnull().sum()

No                          0
Coastal_Cell                0
District                    0
Area_Location               0
Longitude_DMS               0
Latitude_DMS                0
Longitude_DD                0
Latitude_DD                 0
Annual_Erosion_Rate_m_yr    9
Physical_Erosion_Score      9
Economic_Score              0
Total_Score                 0
ISMP_2022_Category          0
NCES_2015_Category          0
Is_Protected                0
Calculated_Total_Score      9
Score_Difference            9
Score_Consistency           0
dtype: int64

In [12]:
df_clean.to_csv(
    "../data/cleaned/jps_coastal_erosion_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [13]:
df_clean["Original_Total_Score"] = df_clean["Total_Score"]

df_clean["Total_Score_Cleaned"] = (
    df_clean["Physical_Erosion_Score"]
    * df_clean["Economic_Score"]
)

df_clean[
    df_clean["No"] == 128
][
    [
        "No",
        "Area_Location",
        "Original_Total_Score",
        "Total_Score_Cleaned"
    ]
]

,No,Area_Location,Original_Total_Score,Total_Score_Cleaned
108,128,Petronas Kertih Refinery,9.3,15.5


In [14]:
Q1 = df_clean["Annual_Erosion_Rate_m_yr"].quantile(0.25)
Q3 = df_clean["Annual_Erosion_Rate_m_yr"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

Q1: 0.9475
Q3: 2.6750000000000003
IQR: 1.7275000000000003
Lower Bound: -1.6437500000000005
Upper Bound: 5.266250000000001


In [15]:
erosion_outliers = df_clean[
    (df_clean["Annual_Erosion_Rate_m_yr"] < lower_bound)
    |
    (df_clean["Annual_Erosion_Rate_m_yr"] > upper_bound)
][
    [
        "No",
        "Coastal_Cell",
        "District",
        "Area_Location",
        "Annual_Erosion_Rate_m_yr"
    ]
]

erosion_outliers

,No,Coastal_Cell,District,Area_Location,Annual_Erosion_Rate_m_yr
15,35,C4,Setiu,Southeast of Kampung Pengkalan Maras,10.18
108,128,C10,Kemaman,Petronas Kertih Refinery,8.19
124,144,C12,Kemaman,Kampung Geliga Cik Akob,10.56


In [16]:
df_clean["Is_Erosion_Outlier"] = (
    (df_clean["Annual_Erosion_Rate_m_yr"] < lower_bound)
    |
    (df_clean["Annual_Erosion_Rate_m_yr"] > upper_bound)
).astype(int)

df_clean["Is_Erosion_Outlier"].value_counts()

Is_Erosion_Outlier
0    122
1      3
Name: count, dtype: int64

In [17]:
df_clean[df_clean["Is_Erosion_Outlier"] == 1][
    [
        "No",
        "Coastal_Cell",
        "Area_Location",
        "Annual_Erosion_Rate_m_yr",
        "Is_Erosion_Outlier"
    ]
]

,No,Coastal_Cell,Area_Location,Annual_Erosion_Rate_m_yr,Is_Erosion_Outlier
15,35,C4,Southeast of Kampung Pengkalan Maras,10.18,1
108,128,C10,Petronas Kertih Refinery,8.19,1
124,144,C12,Kampung Geliga Cik Akob,10.56,1


In [18]:
df_clean["ISMP_2022_Category_Numeric"] = pd.to_numeric(
    df_clean["ISMP_2022_Category"],
    errors="coerce"
)

df_clean[
    df_clean["ISMP_2022_Category_Numeric"].isna()
][
    [
        "No",
        "Area_Location",
        "ISMP_2022_Category",
        "ISMP_2022_Category_Numeric",
        "Is_Protected"
    ]
]

,No,Area_Location,ISMP_2022_Category,ISMP_2022_Category_Numeric,Is_Protected
16,36,North of Kampung Pak Tuyu,Protected,NaN,1
17,37,Kampung Pak Tuyu,Protected,NaN,1
18,38,Kampung Seberang Takir,Protected,NaN,1
90,110,Kampung Sura,Protected,NaN,1
91,111,Kampung Sura,Protected,NaN,1
92,112,Kampung Balai Besar,Protected,NaN,1
100,120,Sand spit near Sungai Paka river mouth,Protected,NaN,1
118,138,Pantai Kemasik (north),Protected,NaN,1
119,139,Kampung Tengah,Protected,NaN,1


In [19]:
print("Dataset shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Protected locations:", df_clean["Is_Protected"].sum())
print("Erosion outliers:", df_clean["Is_Erosion_Outlier"].sum())

print("\nMissing values:")
print(df_clean.isnull().sum())

Dataset shape: (125, 22)
Duplicate rows: 0
Protected locations: 9
Erosion outliers: 3

Missing values:
No                            0
Coastal_Cell                  0
District                      0
Area_Location                 0
Longitude_DMS                 0
Latitude_DMS                  0
Longitude_DD                  0
Latitude_DD                   0
Annual_Erosion_Rate_m_yr      9
Physical_Erosion_Score        9
Economic_Score                0
Total_Score                   0
ISMP_2022_Category            0
NCES_2015_Category            0
Is_Protected                  0
Calculated_Total_Score        9
Score_Difference              9
Score_Consistency             0
Original_Total_Score          0
Total_Score_Cleaned           9
Is_Erosion_Outlier            0
ISMP_2022_Category_Numeric    9
dtype: int64


In [20]:
df_clean = df_clean.drop(
    columns=[
        "Calculated_Total_Score",
        "Score_Difference",
        "Score_Consistency"
    ],
    errors="ignore"
)

In [21]:
df_clean.to_csv(
    "../data/cleaned/jps_coastal_erosion_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [1]:
df_clean["Total_Score_Cleaned"] = np.where(
    df_clean["Is_Protected"] == 1,
    df_clean["Total_Score"],
    df_clean["Physical_Erosion_Score"] * df_clean["Economic_Score"]
)

NameError: name 'np' is not defined

In [2]:
import numpy as np

In [3]:
df_clean["Total_Score_Cleaned"] = np.where(
    df_clean["Is_Protected"] == 1,
    df_clean["Total_Score"],
    df_clean["Physical_Erosion_Score"] * df_clean["Economic_Score"]
)

NameError: name 'df_clean' is not defined

In [4]:
import pandas as pd

In [5]:
df_clean["Total_Score_Cleaned"] = np.where(
    df_clean["Is_Protected"] == 1,
    df_clean["Total_Score"],
    df_clean["Physical_Erosion_Score"] * df_clean["Economic_Score"]
)

NameError: name 'df_clean' is not defined

In [6]:
df = pd.read_csv("../data/raw/jps_coastal_erosion_data.csv")
df_clean = df.copy()

In [7]:
df_clean["Total_Score_Cleaned"] = np.where(
    df_clean["Is_Protected"] == 1,
    df_clean["Total_Score"],
    df_clean["Physical_Erosion_Score"] * df_clean["Economic_Score"]
)

KeyError: 'Is_Protected'

In [8]:
df_clean["Is_Protected"] = (
    df_clean["Annual_Erosion_Rate_m_yr"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("protected")
).astype(int)

In [9]:
df_clean["Annual_Erosion_Rate_m_yr"] = pd.to_numeric(
    df_clean["Annual_Erosion_Rate_m_yr"],
    errors="coerce"
)

df_clean["Physical_Erosion_Score"] = pd.to_numeric(
    df_clean["Physical_Erosion_Score"],
    errors="coerce"
)

df_clean["Economic_Score"] = pd.to_numeric(
    df_clean["Economic_Score"],
    errors="coerce"
)

df_clean["Total_Score"] = pd.to_numeric(
    df_clean["Total_Score"],
    errors="coerce"
)

In [10]:
df_clean["Original_Total_Score"] = df_clean["Total_Score"]

df_clean["Total_Score_Cleaned"] = np.where(
    df_clean["Is_Protected"] == 1,
    df_clean["Total_Score"],
    df_clean["Physical_Erosion_Score"] * df_clean["Economic_Score"]
)

In [11]:
df_clean.to_csv(
    "../data/cleaned/jps_coastal_erosion_cleaned.csv",
    index=False
)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.


In [4]:
print("Total records:", len(df_clean))
print("First JPS No:", df_clean["No"].min())
print("Last JPS No:", df_clean["No"].max())

print("Missing station numbers:",
      sorted(set(range(1, 145)) - set(df_clean["No"])))

print("Duplicate station numbers:",
      df_clean["No"].duplicated().sum())

Total records: 144
First JPS No: 1
Last JPS No: 144
Missing station numbers: []
Duplicate station numbers: 0


In [5]:
print("Total records:", len(df_clean))
print("First JPS No:", df_clean["No"].min())
print("Last JPS No:", df_clean["No"].max())

print("Missing station numbers:",
      sorted(set(range(1, 145)) - set(df_clean["No"])))

print("Duplicate station numbers:",
      df_clean["No"].duplicated().sum())

Total records: 144
First JPS No: 1
Last JPS No: 144
Missing station numbers: []
Duplicate station numbers: 0


In [6]:
df_clean["Is_Protected"] = (
    df_clean["Annual_Erosion_Rate_m_yr"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("protected")
).astype(int)

print(df_clean["Is_Protected"].value_counts())

Is_Protected
0    135
1      9
Name: count, dtype: int64


In [7]:
numeric_columns = [
    "Annual_Erosion_Rate_m_yr",
    "Physical_Erosion_Score",
    "Economic_Score",
    "Total_Score"
]

for col in numeric_columns:
    df_clean[col] = pd.to_numeric(
        df_clean[col],
        errors="coerce"
    )

print(df_clean[numeric_columns].isnull().sum())

Annual_Erosion_Rate_m_yr    9
Physical_Erosion_Score      9
Economic_Score              0
Total_Score                 0
dtype: int64


In [8]:
# Preserve original JPS total scores
df_clean["Original_Total_Score"] = df_clean["Total_Score"].copy()

# Calculate total scores
df_clean["Total_Score_Cleaned"] = (
    df_clean["Physical_Erosion_Score"] *
    df_clean["Economic_Score"]
).round(2)

# Protected locations retain their original zero total score
df_clean.loc[
    df_clean["Is_Protected"] == 1,
    "Total_Score_Cleaned"
] = 0.0

# Find significant score differences
mismatch = df_clean[
    (df_clean["Is_Protected"] == 0) &
    (
        abs(
            df_clean["Original_Total_Score"] -
            df_clean["Total_Score_Cleaned"]
        ) > 0.02
    )
]

print("Number of score mismatches:", len(mismatch))

print(mismatch[
    ["No", "Area_Location", "Original_Total_Score",
     "Total_Score_Cleaned"]
])

Number of score mismatches: 1
      No             Area_Location  Original_Total_Score  Total_Score_Cleaned
127  128  Petronas Kertih Refinery                   9.3                 15.5


In [9]:
# Select erosion rates from non-protected locations
erosion_data = df_clean.loc[
    df_clean["Is_Protected"] == 0,
    "Annual_Erosion_Rate_m_yr"
].dropna()

# Calculate IQR
Q1 = erosion_data.quantile(0.25)
Q3 = erosion_data.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify outliers
df_clean["Is_Erosion_Outlier"] = (
    (df_clean["Is_Protected"] == 0) &
    (
        (df_clean["Annual_Erosion_Rate_m_yr"] < lower_bound) |
        (df_clean["Annual_Erosion_Rate_m_yr"] > upper_bound)
    )
)

outliers = df_clean[df_clean["Is_Erosion_Outlier"]]

print("Lower bound:", round(lower_bound, 2))
print("Upper bound:", round(upper_bound, 2))
print("Number of outliers:", len(outliers))

print(outliers[
    ["No", "Area_Location", "Annual_Erosion_Rate_m_yr"]
])

Lower bound: -1.77
Upper bound: 5.53
Number of outliers: 10
      No                         Area_Location  Annual_Erosion_Rate_m_yr
16    17     North of Sungai Setiu river mouth                      7.32
17    18              Sungai Setiu river mouth                     15.52
18    19              Sungai Setiu river mouth                      7.09
19    20              Sungai Setiu river mouth                      7.52
20    21           Sand spit near Sungai Setiu                      8.76
21    22     West of Sungai Merang river mouth                     12.59
33    34                  Kampung Padang Maras                      7.85
34    35  Southeast of Kampung Pengkalan Maras                     10.18
127  128              Petronas Kertih Refinery                      8.19
143  144               Kampung Geliga Cik Akob                     10.56


In [10]:
print("=== FINAL DATA CLEANING CHECK ===")

print("Dataset shape:", df_clean.shape)

print("\nProtected locations:")
print(df_clean["Is_Protected"].value_counts())

print("\nMissing values:")
print(df_clean[
    ["Annual_Erosion_Rate_m_yr",
     "Physical_Erosion_Score",
     "Economic_Score",
     "Total_Score_Cleaned"]
].isnull().sum())

print("\nDuplicate records:", df_clean.duplicated().sum())

print("\nOutliers:", df_clean["Is_Erosion_Outlier"].sum())

print("\nNon-protected missing values:")
print(df_clean.loc[
    df_clean["Is_Protected"] == 0,
    ["Annual_Erosion_Rate_m_yr",
     "Physical_Erosion_Score",
     "Economic_Score"]
].isnull().sum())

=== FINAL DATA CLEANING CHECK ===
Dataset shape: (144, 18)

Protected locations:
Is_Protected
0    135
1      9
Name: count, dtype: int64

Missing values:
Annual_Erosion_Rate_m_yr    9
Physical_Erosion_Score      9
Economic_Score              0
Total_Score_Cleaned         0
dtype: int64

Duplicate records: 0

Outliers: 10

Non-protected missing values:
Annual_Erosion_Rate_m_yr    0
Physical_Erosion_Score      0
Economic_Score              0
dtype: int64


In [11]:
import os

# Create cleaned folder if needed
os.makedirs("../data/cleaned", exist_ok=True)

# Save the updated dataset
output_path = "../data/cleaned/jps_coastal_erosion_cleaned_144.csv"

df_clean.to_csv(output_path, index=False)

# Verify the saved file
df_check = pd.read_csv(output_path)

print("Cleaned dataset saved successfully!")
print("Dataset shape:", df_check.shape)
print("File location:", output_path)

Cleaned dataset saved successfully!
Dataset shape: (144, 18)
File location: ../data/cleaned/jps_coastal_erosion_cleaned_144.csv
